# C3S Land Cover Change Analysis — Yellowstone to Yukon (2000–2022)

This notebook analyzes land cover and land use change across the Yellowstone to Yukon (Y2Y) region from 2000 to 2022, using the Copernicus Climate Change Service (C3S) Annual Land Cover dataset hosted on the GEE Community Catalog.

**Dataset:** C3S Land Cover Classification Gridded Maps — Annual global land cover maps at 300 m resolution.  
**GEE Collection ID:** `projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS`  
**Key band:** `b1` — Land cover classification (0–220, LCCS scheme) — Land cover classification (0–220, LCCS scheme)  
**Source sensors:** SPOT-VGT (1999–2013) → PROBA-V (2014–2019) → Sentinel-3 OLCI/SLSTR (2020+)  
**Available years in sat-io catalog:** 2000–2022

The C3S dataset uses the identical UN LCCS class legend as the ESA CCI LC archive, so class definitions, colors, and aggregations are fully consistent.

**Workflow:**
1. Load C3S LC collection and Y2Y boundary
2. Inspect available years and band metadata
3. Visualize land cover maps interactively
4. Calculate area (km²) per land cover class per year across Y2Y
5. Plot time-series of class-level area change
6. Detect net change: 2000 vs. 2022
7. Build a land cover transition matrix between two epochs
8. Forest cover trend analysis
9. Export results to Google Drive

## 1. Load packages and initialize GEE

In [76]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

In [77]:
ee.Authenticate()

True

In [78]:
ee.Initialize(project='y2y-climate-benefits')

## 2. Define datasets

In [79]:
# ── C3S Land Cover annual collection (GEE Community Catalog) ─────────────────
# Covers 1992–2022 at 300 m; uses the same LCCS legend as ESA CCI LC.
lc_collection = ee.ImageCollection('projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS')

# ── Y2Y boundary ──────────────────────────────────────────────────────────────
y2y = ee.FeatureCollection('projects/ee-bermane/assets/y2y')
y2y_geom = y2y.geometry()

# ── Temporal range of interest ────────────────────────────────────────────────
START_YEAR = 2000   # earliest year available in sat-io C3S catalog ingestion
END_YEAR   = 2022   # last year in C3S collection

lc_filtered = (
    lc_collection
    .filter(ee.Filter.calendarRange(START_YEAR, END_YEAR, 'year'))
)

print('Images in collection:', lc_filtered.size().getInfo())

Images in collection: 23


In [80]:
# Inspect the first image to confirm band name and time structure
first_img = lc_filtered.first()
print('Band names:',  first_img.bandNames().getInfo())
print('First year:',  ee.Date(first_img.get('system:time_start')).format('YYYY').getInfo())

# Sorted list of all available years in the filtered collection
available_years = (
    lc_filtered
    .aggregate_array('system:time_start')
    .map(lambda t: ee.Date(t).get('year'))
    .sort()
    .getInfo()
)
print('Available years:', available_years)

Band names: ['b1']
First year: 2000
Available years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


## 3. C3S Land Cover class definitions

The C3S LC classification follows the UN LCCS (Land Cover Classification System) with 37 classes coded 0–220 — identical to the ESA CCI LC legend. Below we define both the full detailed class lookup and a set of 9 broad aggregated types suited to regional-scale ecological interpretation over Y2Y.

In [81]:
# C3S LC class values, names, and official hex colors
# Source: GEE Community Catalog (gee-community-catalog.org/projects/c3slc/)
LC_CLASSES = {
    10:  ("Cropland, rainfed",                                 "#ffff64"),
    11:  ("Cropland, rainfed — herbaceous cover",              "#ffff64"),
    12:  ("Cropland, rainfed — tree/shrub cover",              "#ffff64"),
    20:  ("Cropland, irrigated / post-flooding",               "#aaf0f0"),
    30:  ("Mosaic cropland >50% / nat. veg. <50%",             "#dcf064"),
    40:  ("Mosaic nat. veg. >50% / cropland <50%",             "#c8c864"),
    50:  ("Tree cover, broadleaved, evergreen",                "#006400"),
    60:  ("Tree cover, broadleaved, deciduous",                "#00a000"),
    61:  ("Tree cover, broadleaved, deciduous, closed",        "#00a000"),
    62:  ("Tree cover, broadleaved, deciduous, open",          "#00a000"),
    70:  ("Tree cover, needleleaved, evergreen",               "#003c00"),
    71:  ("Tree cover, needleleaved, evergreen, closed",       "#003c00"),
    72:  ("Tree cover, needleleaved, evergreen, open",         "#003c00"),
    80:  ("Tree cover, needleleaved, deciduous",               "#285000"),
    81:  ("Tree cover, needleleaved, deciduous, closed",       "#285000"),
    82:  ("Tree cover, needleleaved, deciduous, open",         "#285000"),
    90:  ("Tree cover, mixed leaf type",                       "#788200"),
    100: ("Mosaic tree & shrub >50% / herbaceous <50%",        "#8ca000"),
    110: ("Mosaic herbaceous >50% / tree & shrub <50%",        "#be9600"),
    120: ("Shrubland",                                         "#966400"),
    121: ("Shrubland, evergreen",                              "#966400"),
    122: ("Shrubland, deciduous",                              "#966400"),
    130: ("Grassland",                                         "#ffb432"),
    140: ("Lichens and mosses",                                "#ffdcd2"),
    150: ("Sparse vegetation (<15%)",                          "#ffebaf"),
    151: ("Sparse tree (<15%)",                                "#ffebaf"),
    152: ("Sparse shrub (<15%)",                               "#ffebaf"),
    153: ("Sparse herbaceous (<15%)",                          "#ffebaf"),
    160: ("Tree cover, flooded, fresh/brackish water",         "#00785a"),
    170: ("Tree cover, flooded, saline water",                 "#009678"),
    180: ("Shrub/herbaceous cover, flooded",                   "#00dc82"),
    190: ("Urban areas",                                       "#c31400"),
    200: ("Bare areas",                                        "#fff5d7"),
    201: ("Consolidated bare areas",                           "#fff5d7"),
    202: ("Unconsolidated bare areas",                         "#fff5d7"),
    210: ("Water bodies",                                      "#0046c8"),
    220: ("Permanent snow and ice",                            "#ffffff"),
}

lc_values = list(LC_CLASSES.keys())
lc_names  = [v[0] for v in LC_CLASSES.values()]
lc_colors = [v[1] for v in LC_CLASSES.values()]

print(f"{len(LC_CLASSES)} LC classes defined.")

37 LC classes defined.


In [82]:
# Broad aggregated classes — ecologically meaningful groupings for Y2Y
BROAD_CLASSES = {
    'Forest':    {'values': [50,60,61,62,70,71,72,80,81,82,90,100], 'color': '#006400', 'value': 1},
    'Shrubland': {'values': [110,120,121,122],                       'color': '#966400', 'value': 2},
    'Grassland': {'values': [130,140,150,151,152,153],               'color': '#ffb432', 'value': 3},
    'Cropland':  {'values': [10,11,12,20,30,40],                    'color': '#ffff64', 'value': 4},
    'Wetland':   {'values': [160,170,180],                          'color': '#00dc82', 'value': 5},
    'Urban':     {'values': [190],                                  'color': '#c31400', 'value': 6},
    'Barren':    {'values': [200,201,202],                          'color': '#fff5d7', 'value': 7},
    'Water':     {'values': [210],                                  'color': '#0046c8', 'value': 8},
    'Snow/Ice':  {'values': [220],                                  'color': '#ffffff', 'value': 9},
}

# Build remap input/output lists for ee.Image.remap()
remap_from, remap_to = [], []
for info in BROAD_CLASSES.values():
    for v in info['values']:
        remap_from.append(v)
        remap_to.append(info['value'])

broad_values = [info['value'] for info in BROAD_CLASSES.values()]
broad_names  = list(BROAD_CLASSES.keys())
broad_colors = [info['color'] for info in BROAD_CLASSES.values()]

print('Broad class mapping ready.')
print(f'remap_from ({len(remap_from)} values):', remap_from)
print(f'remap_to   ({len(remap_to)} values):', remap_to)

Broad class mapping ready.
remap_from (37 values): [50, 60, 61, 62, 70, 71, 72, 80, 81, 82, 90, 100, 110, 120, 121, 122, 130, 140, 150, 151, 152, 153, 10, 11, 12, 20, 30, 40, 160, 170, 180, 190, 200, 201, 202, 210, 220]
remap_to   (37 values): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 5, 5, 5, 6, 7, 7, 7, 8, 9]


## 4. Visualize land cover maps

In [83]:
# Pull first (START_YEAR) and last (END_YEAR) images
lc_start = lc_filtered.filter(ee.Filter.calendarRange(START_YEAR, START_YEAR, 'year')).first()
lc_last  = lc_filtered.filter(ee.Filter.calendarRange(END_YEAR,   END_YEAR,   'year')).first()

# Remap to broad classes
broad_start = lc_start.select('b1').remap(remap_from, remap_to).rename('broad_lc')
broad_last  = lc_last.select('b1').remap(remap_from, remap_to).rename('broad_lc')

# Visualization parameters — min=1/max=9 so GEE maps each integer exactly to one palette color
broad_vis = {'min': 1, 'max': 9, 'palette': broad_colors}

# Interactive map
Map = geemap.Map()
Map.centerObject(y2y_geom, zoom=5)

Map.addLayer(broad_start.clipToCollection(y2y), broad_vis, f'LC {START_YEAR} (broad)', True,  0.9)
Map.addLayer(broad_last.clipToCollection(y2y),  broad_vis, f'LC {END_YEAR} (broad)',   False, 0.9)
Map.addLayer(
    lc_start.select('b1').clipToCollection(y2y),
    {'min': 10, 'max': 220, 'palette': lc_colors},
    f'LC {START_YEAR} (detailed)', False, 0.9
)
Map.addLayer(
    y2y.style(**{'color': 'black', 'fillColor': '00000000', 'width': 2}),
    {}, 'Y2Y boundary'
)

legend_dict = {name: info['color'] for name, info in BROAD_CLASSES.items()}
Map.add_legend(title='Land Cover', legend_dict=legend_dict, position='bottomright')

Map

Map(center=[55.5658848933782, -121.61243738923375], controls=(WidgetControl(options=['position', 'transparent_…

## 5. Calculate area (km²) per land cover class per year

We use `ee.Image.pixelArea()` combined with a grouped reducer to sum actual pixel areas per class. This handles the non-equal-area projection correctly at the 300 m scale. One GEE call per year is made client-side for readability; for very long time series consider batching with `ee.ImageCollection.map()`.

In [ ]:
def compute_lc_area(year):
    """Return GEE groups list: [{lc_class, sum_m2}] for the given year clipped to Y2Y."""
    img = (
        lc_filtered
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .first()
        .select('b1')
        .clipToCollection(y2y)
    )
    lc_with_area = img.addBands(ee.Image.pixelArea().rename('area'))
    stats = lc_with_area.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=0, groupName='lc_class'),
        geometry=y2y_geom,
        scale=300,
        maxPixels=1e13,
        bestEffort=True,
    )
    return stats.get('groups')


print('Computing area statistics per year — this takes ~1–2 min...')
rows = []
for yr in available_years:
    groups = compute_lc_area(yr).getInfo()
    for g in groups:
        rows.append({
            'year':     yr,
            'lc_class': int(g['lc_class']),
            'area_m2':  g['sum'],
        })
    print(f'  {yr} ✓')

df_raw = pd.DataFrame(rows)
df_raw['area_km2'] = df_raw['area_m2'] / 1e6
print(f'\nDone. {len(df_raw)} class-year records.')

In [ ]:
# Add class names and aggregate to broad classes
df_raw['lc_name'] = df_raw['lc_class'].map(
    lambda v: LC_CLASSES.get(v, (f'Class {v}', '#aaaaaa'))[0]
)

df_raw['broad_class'] = df_raw['lc_class'].map(
    lambda v: next((b for b, info in BROAD_CLASSES.items() if v in info['values']), 'No data')
)

# Broad-class pivot: rows = year, columns = broad class name
df_broad = (
    df_raw
    .groupby(['year', 'broad_class'])['area_km2']
    .sum()
    .unstack('broad_class')
    .fillna(0)
    .sort_index()
)

# Detailed-class pivot
df_detailed = (
    df_raw
    .pivot_table(index='year', columns='lc_name', values='area_km2', aggfunc='sum')
    .fillna(0)
    .sort_index()
)

print('Total Y2Y area check (km²) — should be roughly constant:')
print(df_broad.sum(axis=1).astype(int))

## 6. Time-series plots — area by land cover class

In [ ]:
# Stacked area chart — broad classes (exclude 'No data')
plot_cols  = [c for c in broad_names if c != 'No data' and c in df_broad.columns]
plot_data  = df_broad[plot_cols]
plot_clrs  = [BROAD_CLASSES[c]['color'] for c in plot_cols]

fig, ax = plt.subplots(figsize=(13, 6))
ax.stackplot(
    plot_data.index, [plot_data[c] for c in plot_cols],
    labels=plot_cols, colors=plot_clrs, alpha=0.85
)
ax.set_title(f'Land Cover Area — Yellowstone to Yukon ({START_YEAR}–{END_YEAR})',
             fontsize=14, pad=12)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Area (km²)', fontsize=11)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlim(plot_data.index.min(), plot_data.index.max())
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig('lc_stacked_area_y2y.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Line chart — area change relative to START_YEAR baseline
yr_start = df_broad.index.min()
yr_end   = df_broad.index.max()

baseline  = df_broad.loc[yr_start, plot_cols]
df_delta  = df_broad[plot_cols].subtract(baseline)

fig, ax = plt.subplots(figsize=(13, 6))
for col in plot_cols:
    ax.plot(df_delta.index, df_delta[col],
            label=col, color=BROAD_CLASSES[col]['color'],
            linewidth=2, marker='o', markersize=4)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title(f'Land Cover Change from {yr_start} Baseline — Y2Y', fontsize=14, pad=12)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel(f'Change from {yr_start} (km²)', fontsize=11)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:+,.0f}'))
ax.set_xlim(df_delta.index.min(), df_delta.index.max())
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(f'lc_delta_from_{yr_start}_y2y.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table: START_YEAR vs. final year
summary = pd.DataFrame({
    f'{yr_start} (km²)': df_broad.loc[yr_start, plot_cols],
    f'{yr_end} (km²)':   df_broad.loc[yr_end,   plot_cols],
})
summary['Change (km²)'] = summary[f'{yr_end} (km²)'] - summary[f'{yr_start} (km²)']
summary['Change (%)']   = (summary['Change (km²)'] / summary[f'{yr_start} (km²)'] * 100).round(2)
summary = summary.sort_values('Change (km²)')

print(f'Land Cover Change Summary — {yr_start} to {yr_end}:')
print(summary.round(1).to_string())

## 7. Change detection — 2000 vs. most recent year

In [ ]:
# Binary change map: did broad class change between START_YEAR and END_YEAR?
change_binary = broad_last.neq(broad_start).rename('changed').clipToCollection(y2y)

# Compute changed and total areas
changed_area_m2 = (
    change_binary
    .multiply(ee.Image.pixelArea())
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=y2y_geom,
        scale=300,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo()['changed']
)

total_area_m2 = (
    ee.Image.pixelArea()
    .clipToCollection(y2y)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=y2y_geom,
        scale=300,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo()['area']
)

pct_changed = changed_area_m2 / total_area_m2 * 100

print(f'Total Y2Y area:   {total_area_m2/1e6:>12,.0f} km²')
print(f'Area changed:     {changed_area_m2/1e6:>12,.0f} km²')
print(f'Percent changed:  {pct_changed:>11.2f}%  ({START_YEAR} → {END_YEAR})')

In [ ]:
# Interactive map: binary change + side-by-side broad LC
Map2 = geemap.Map()
Map2.centerObject(y2y_geom, zoom=5)

Map2.addLayer(broad_start.clipToCollection(y2y), broad_vis,
              f'Broad LC {START_YEAR}',            False, 0.85)
Map2.addLayer(broad_last.clipToCollection(y2y),  broad_vis,
              f'Broad LC {END_YEAR}',             False, 0.85)
Map2.addLayer(change_binary,
              {'min': 0, 'max': 1, 'palette': ['#d4d4d4', '#e63946']},
              f'Changed {START_YEAR}→{END_YEAR}',  True,  0.8)
Map2.addLayer(
    y2y.style(**{'color': 'black', 'fillColor': '00000000', 'width': 2}),
    {}, 'Y2Y boundary'
)

Map2.add_legend(
    title='Change',
    legend_dict={'No change': '#d4d4d4', 'Changed': '#e63946'},
    position='bottomright'
)
Map2

## 8. Land cover transition matrix

Shows the area (km²) that moved from each broad class in the start year (2000) to each broad class in the final year (2022). Diagonal = stable; off-diagonal = conversions.

In [ ]:
def compute_transition_matrix(img_from, img_to, geom, scale=300):
    """
    Compute transition matrix (area km²) between two broad-class LC images.
    Returns a DataFrame: rows = 'from' class (START_YEAR), columns = 'to' class (END_YEAR).
    """
    val_to_name = {info['value']: name for name, info in BROAD_CLASSES.items()}
    names_ordered = [n for n in broad_names if n != 'No data']

    # Encode from*100 + to in one band, then sum pixel areas per code
    combined = img_from.multiply(100).add(img_to).rename('transition')
    combined_area = combined.addBands(ee.Image.pixelArea().rename('area'))

    result = (
        combined_area
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=0, groupName='transition'),
            geometry=geom,
            scale=scale,
            maxPixels=1e13,
            bestEffort=True,
        )
        .get('groups')
        .getInfo()
    )

    matrix = pd.DataFrame(0.0, index=names_ordered, columns=names_ordered)
    for entry in result:
        code      = int(entry['transition'])
        from_val  = code // 100
        to_val    = code % 100
        from_name = val_to_name.get(from_val)
        to_name   = val_to_name.get(to_val)
        if from_name and to_name and from_name in matrix.index and to_name in matrix.columns:
            matrix.loc[from_name, to_name] += entry['sum'] / 1e6  # m² → km²

    return matrix


print('Computing transition matrix...')
transition_df = compute_transition_matrix(broad_start, broad_last, y2y_geom)
print('Done.')

In [ ]:
# Plot transition matrix heatmap (log-scaled color)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(np.log1p(transition_df.values), cmap='YlOrRd', aspect='auto')

n = len(transition_df)
ax.set_xticks(range(n));  ax.set_yticks(range(n))
ax.set_xticklabels(transition_df.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(transition_df.index, fontsize=9)
ax.set_xlabel(f'Land Cover {END_YEAR}', fontsize=11)
ax.set_ylabel(f'Land Cover {START_YEAR}', fontsize=11)
ax.set_title(
    f'Land Cover Transition Matrix — Y2Y ({START_YEAR} → {END_YEAR})\n(color scale: log₁(area km² + 1))',
    fontsize=12, pad=10
)

# Annotate each cell
max_val = transition_df.values.max()
for i in range(n):
    for j in range(n):
        val = transition_df.iloc[i, j]
        if val > 0:
            txt_color = 'white' if val > max_val * 0.5 else 'black'
            ax.text(j, i, f'{val:,.0f}', ha='center', va='center',
                    fontsize=6.5, color=txt_color)

plt.colorbar(im, ax=ax, label='log(area km² + 1)', shrink=0.7)
plt.tight_layout()
plt.savefig('lc_transition_matrix_y2y.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Net gains, losses, and net change per broad class
diag   = np.diag(transition_df.values)
gains  = transition_df.sum(axis=0) - diag   # incoming from other classes
losses = transition_df.sum(axis=1) - diag   # outgoing to other classes
net    = pd.Series(gains.values - losses.values, index=transition_df.columns)

net_df = pd.DataFrame({
    'Gross gain (km²)':  gains.values,
    'Gross loss (km²)':  losses.values,
    'Net change (km²)':  net.values,
}, index=transition_df.columns).sort_values('Net change (km²)')

print(f'Net land cover gains/losses — {yr_start} to {yr_end}:')
print(net_df.round(1).to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
bar_clrs = ['#2a9d8f' if v >= 0 else '#e76f51' for v in net_df['Net change (km²)']]
ax.barh(net_df.index, net_df['Net change (km²)'], color=bar_clrs, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Net Area Change (km²)', fontsize=11)
ax.set_title(f'Net Land Cover Change — Y2Y ({yr_start}→{yr_end})', fontsize=12, pad=10)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:+,.0f}'))
plt.tight_layout()
plt.savefig('lc_net_change_bar_y2y.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Forest cover trend

Forest is the dominant and ecologically most critical class in Y2Y. We plot its annual area with a linear trend line and also break out needleleaved vs. broadleaved forest sub-types.

In [ ]:
# Total forest trend with linear fit
forest_series = df_broad['Forest']
years_arr = forest_series.index.values.astype(float)
area_arr  = forest_series.values

m, b = np.polyfit(years_arr, area_arr, 1)
net_forest = area_arr[-1] - area_arr[0]

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(years_arr, area_arr, alpha=0.2, color='#006400')
ax.plot(years_arr, area_arr, 'o-', color='#006400',
        linewidth=2, markersize=5, label='Total forest')
ax.plot(years_arr, m * years_arr + b, '--', color='black', linewidth=1.2,
        label=f'Linear trend: {m:+,.0f} km²/yr')

ax.set_title(f'Forest Cover Trend — Yellowstone to Yukon ({START_YEAR}–{END_YEAR})',
             fontsize=13, pad=10)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Forest Area (km²)', fontsize=11)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('forest_trend_y2y.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Forest area {yr_start}:  {area_arr[0]:>12,.0f} km²')
print(f'Forest area {yr_end}:    {area_arr[-1]:>12,.0f} km²')
print(f'Net change:             {net_forest:>+12,.0f} km²  ({net_forest/area_arr[0]*100:+.2f}%)')

In [ ]:
# Forest sub-type breakdown: needleleaved vs. broadleaved vs. mixed
needle_classes = [70, 71, 72, 80, 81, 82]
broad_lv_classes = [50, 60, 61, 62]
mixed_classes = [90, 100]

def class_area_series(class_list, df):
    """Sum area across a list of LC class names for all years."""
    names = [LC_CLASSES[v][0] for v in class_list if LC_CLASSES[v][0] in df.columns]
    return df[names].sum(axis=1) if names else pd.Series(0, index=df.index)

needle_series  = class_area_series(needle_classes,   df_detailed)
broadlv_series = class_area_series(broad_lv_classes, df_detailed)
mixed_series   = class_area_series(mixed_classes,    df_detailed)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(needle_series.index,  needle_series,  'o-', color='#003c00', linewidth=2,
        markersize=4, label='Needleleaved')
ax.plot(broadlv_series.index, broadlv_series, 's-', color='#00a000', linewidth=2,
        markersize=4, label='Broadleaved')
ax.plot(mixed_series.index,   mixed_series,   '^-', color='#788200', linewidth=2,
        markersize=4, label='Mixed / Mosaic')

ax.set_title(f'Forest Sub-Type Trends — Y2Y ({START_YEAR}–{END_YEAR})', fontsize=13, pad=10)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Area (km²)', fontsize=11)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('forest_subtype_trend_y2y.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export results to Google Drive

In [ ]:
# Save area statistics CSVs locally
df_broad.to_csv(f'lc_area_broad_class_y2y_{START_YEAR}_{END_YEAR}.csv')
df_detailed.to_csv(f'lc_area_detailed_class_y2y_{START_YEAR}_{END_YEAR}.csv')
transition_df.to_csv(f'lc_transition_matrix_y2y_{START_YEAR}_{END_YEAR}.csv')
net_df.to_csv(f'lc_net_gains_losses_y2y_{START_YEAR}_{END_YEAR}.csv')
print('CSVs saved.')

In [ ]:
# Export GeoTIFFs to Google Drive (folder: y2y_lc_change)
DRIVE_FOLDER = 'y2y_lc_change'

exports = [
    (broad_start.unmask(255).toByte(),   f'y2y_broad_lc_{START_YEAR}'),
    (broad_last.unmask(255).toByte(),    f'y2y_broad_lc_{END_YEAR}'),
    (change_binary.unmask(255).toByte(), f'y2y_lc_change_binary_{START_YEAR}_{END_YEAR}'),
]

tasks = []
for img, name in exports:
    task = ee.batch.Export.image.toDrive(
        image=img.clipToCollection(y2y),
        description=name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=name,
        region=y2y_geom,
        scale=300,
        crs='EPSG:4326',
        maxPixels=1e13,
    )
    task.start()
    tasks.append(task)
    print(f'Submitted: {name}  →  {task.status()["state"]}')

In [ ]:
# Monitor export task status
for task in tasks:
    s = task.status()
    print(f'{s[\"description\"]:55s}  {s[\"state\"]}')